# §1.8.3 — 선형 조각을 실제로 세어 보기

> 딥러닝 교재 · 1부 1장 8절 3항 (🐍)
> 선행: §1.7.4(톱니 손계산) · §1.7.6(표현과 학습의 간극) · §1.8.1(상한) · §1.8.2(하한)

## 이 노트북이 답하는 질문

1. **§1.8.1의 정리가 맞는가?** 톱니 가중치를 심어 조각 수가 정확히 $2^L$인지 센다.
2. **깊이를 늘리면 조각 수가 지수적으로 느는가?** 무작위 초기화와 학습된 망에서 실제로 센다.
3. **§1.7.6이 예고한 것 — 표현할 수 있는데 학습하지 못하는가?** 심은 해가 손실 0인데 경사하강이 도달하는지 본다.

**예상 실행 시간** CPU 단일 코어 약 90초 (`FAST = True`이면 약 30초).

세 질문의 답이 각각 **그렇다 · 아니다 · 그렇다(못 한다)** 이며, 두 번째와 세 번째가 §1.8.4–1.8.5의 재료가 된다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260803
NGRID    = 100_001    # 조각 세기용 격자
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_1_8_3_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 망과 조각 세기

**망.** 입력 1차원, 은닉층 $L$개(각 폭 $w$), 출력 1차원인 ReLU 완전연결망.

**조각 세는 법.** ReLU 망의 출력은 **정확히** 조각선형이므로, 출력의 기울기가 바뀌는 지점을 세면 된다.
격자 위에서 차분 기울기를 구하고, 값이 뛰는 곳을 꺾임으로 센다.

> ⚠︎ 두 가지 수치적 주의가 필요하다. 꺾임이 격자점 **사이**에 떨어지면 차분 기울기가 중간값을 갖게 되어
> 한 꺾임이 두 번 세어진다 — 연속된 검출 위치를 하나로 **병합**해야 한다.
> 그리고 격자 간격보다 좁은 조각은 놓치므로, 이 방법은 항상 **하한**이다.

In [ ]:
def init_net(L, w, rng, bstd=0.5):
    # 은닉층 L개, 폭 w. He 초기화. 편향은 은닉층에만 무작위로 준다.
    dims = [1] + [w]*L + [1]
    Ws, bs = [], []
    for i in range(len(dims)-1):
        Ws.append(rng.normal(0, np.sqrt(2.0/dims[i]), (dims[i], dims[i+1])))
        bs.append(rng.normal(0, bstd, dims[i+1]) if i < len(dims)-2 else np.zeros(dims[i+1]))
    return Ws, bs

def forward(x, Ws, bs):
    a = np.asarray(x, float).reshape(-1, 1)
    for i in range(len(Ws)-1):
        a = np.maximum(a @ Ws[i] + bs[i], 0)
    return (a @ Ws[-1] + bs[-1]).ravel()

def count_regions(Ws, bs, xs):
    y = forward(xs, Ws, bs)
    s = np.diff(y) / np.diff(xs)                       # 구간별 기울기
    d = np.abs(np.diff(s))
    sc = np.maximum(np.abs(s[:-1]), np.abs(s[1:]))
    idx = np.flatnonzero(d > 1e-6*(1.0 + sc))          # 기울기가 뛰는 위치
    if len(idx) == 0:
        return 1
    return int(np.sum(np.diff(idx) > 1)) + 2           # 연속 위치는 한 꺾임으로 병합

XS = np.linspace(0, 1, NGRID)
print(f"격자 {NGRID:,}점, 간격 {1/(NGRID-1):.2e}")

### 1.1 §1.8.1의 정리 확인 — 톱니를 심는다

§1.7.4에서 손으로 만든 가중치를 그대로 넣는다. 각 층이 폭 2이고

$$T(x) = 2\,\mathrm{ReLU}(x) - 4\,\mathrm{ReLU}(x - 1/2)$$

를 구현하도록 쌓으면, 깊이 $L$인 망이 $T_L$을 **정확히** 계산한다.

In [ ]:
def plant_sawtooth(L):
    # 깊이 L, 폭 2로 T_L 을 정확히 구현하는 가중치
    Ws = [np.array([[1.0, 1.0]])]
    bs = [np.array([0.0, -0.5])]
    for _ in range(L-1):
        Ws.append(np.array([[2.0, 2.0], [-4.0, -4.0]]))
        bs.append(np.array([0.0, -0.5]))
    Ws.append(np.array([[2.0], [-4.0]]))
    bs.append(np.array([0.0]))
    return Ws, bs

def T_ref(x, L):
    z = np.asarray(x, float)
    for _ in range(L):
        z = np.where(z <= 0.5, 2*z, 2 - 2*z)
    return z

LMAX_PLANT = 8 if FAST else 11
print("  L    조각수   이론 2^L    최대오차")
plant_counts = []
for L in range(1, LMAX_PLANT+1):
    Ws, bs = plant_sawtooth(L)
    c = count_regions(Ws, bs, XS)
    err = float(np.abs(forward(XS, Ws, bs) - T_ref(XS, L)).max())
    plant_counts.append(c)
    print(f" {L:>2}   {c:>6}   {2**L:>8}    {err:.1e}")
    assert c == 2**L, (L, c)
print("\n§1.8.1의 정리가 L={0}까지 정확히 확인되었다. 유닛 {1}개로 조각 {2}개.".format(
      LMAX_PLANT, 2*LMAX_PLANT, 2**LMAX_PLANT))

In [ ]:
xz = np.linspace(0, 1, 2001)
fig, axes = plt.subplots(1, 4, figsize=(10.2, 2.5), sharey=True)
for ax, L in zip(axes, [1, 2, 3, 4]):
    Ws, bs = plant_sawtooth(L)
    ax.plot(xz, forward(xz, Ws, bs), color=CB[5], lw=1.2)
    ax.set_title(lab(f'$T_{{{L}}}$ — 조각 {2**L}개', f'$T_{{{L}}}$ — {2**L} pieces'), fontsize=10)
    ax.set_xlabel('$x$')
axes[0].set_ylabel('$T_L(x)$')
fig.suptitle(lab('층을 하나 쌓을 때마다 조각이 두 배가 된다 (폭 2 고정)',
                 'each added layer doubles the pieces (width fixed at 2)'), y=1.06, fontsize=10)
show('sawtooth_shapes')

---
## 2. 무작위 초기화의 조각 수

§1.8.1 4절에서 일반 망의 상한이 **깊이에 지수적**이라고 했다. 1차원에서 그 상한은 $(w+1)^L$ 이다.

**상한이 달성되는가?** 1.1절의 톱니는 달성한다 — 그러나 그것은 달성하도록 **설계된** 가중치다.
아무 가중치나 뽑으면 어떻게 되는지 본다.

In [ ]:
WS_LIST = [4, 16]
LS = [1, 2, 3, 4, 6, 8]
NSEED = 3 if FAST else 5

init_counts = {}
print("       무작위 초기화 조각 수")
print("  w   L    실측    총 유닛 wL    이론 최대 (w+1)^L")
for w in WS_LIST:
    init_counts[w] = []
    for L in LS:
        c = [count_regions(*init_net(L, w, np.random.default_rng(SEED+s)), XS)
             for s in range(NSEED)]
        init_counts[w].append(float(np.mean(c)))
        print(f" {w:>2}  {L:>2}   {np.mean(c):7.1f}    {w*L:>9}    {(w+1)**L:>16,}")

> **읽는 법.** 실측 조각 수가 **총 유닛 수 $wL$ 정도**에서 맴돈다. 이론 최대 $(w+1)^L$ 과는 비교가 안 된다.
> $w = 16$, $L = 8$ 이면 상한이 70억인데 실제로는 수십 개다.
>
> 곧 **깊이에 대한 지수적 표현력은 특별히 설계했을 때만 나타난다.** 무작위로 뽑은 망에는 없다.

---
## 3. 학습하면 달라지는가

학습이 조각 수를 지수적으로 끌어올리는지 본다. 매끄러운 목표 함수를 주고 깊이를 바꿔 학습시킨다.

In [ ]:
def train(Ws, bs, x, y, steps, lr=3e-3):
    mW=[np.zeros_like(W) for W in Ws]; vW=[np.zeros_like(W) for W in Ws]
    mb=[np.zeros_like(b) for b in bs]; vb=[np.zeros_like(b) for b in bs]
    n = len(x); best = np.inf; hist = []
    for t in range(1, steps+1):
        a = x.reshape(-1,1); acts=[a]; pre=[]
        for i in range(len(Ws)-1):
            z = a @ Ws[i] + bs[i]; pre.append(z); a = np.maximum(z, 0); acts.append(a)
        out = (a @ Ws[-1] + bs[-1]).ravel()
        d = out - y
        loss = float(np.mean(d**2)); best = min(best, loss); hist.append(loss)
        g = (2.0/n) * d.reshape(-1,1)
        gW=[None]*len(Ws); gb=[None]*len(bs)
        gW[-1] = acts[-1].T @ g; gb[-1] = g.sum(0)
        delta = g @ Ws[-1].T
        for i in range(len(Ws)-2, -1, -1):
            delta = delta * (pre[i] > 0)
            gW[i] = acts[i].T @ delta; gb[i] = delta.sum(0)
            if i > 0:
                delta = delta @ Ws[i].T
        for i in range(len(Ws)):                       # Adam
            for (p, gp, m, v) in ((Ws[i], gW[i], mW, vW), (bs[i], gb[i], mb, vb)):
                m[i] = 0.9*m[i] + 0.1*gp
                v[i] = 0.999*v[i] + 0.001*gp**2
                p -= lr * (m[i]/(1-0.9**t)) / (np.sqrt(v[i]/(1-0.999**t)) + 1e-8)
    return best, hist

rng = np.random.default_rng(SEED)
x_tr = np.sort(rng.uniform(0, 1, 600 if FAST else 800))
y_smooth = np.sin(3*np.pi*x_tr) + 0.4*np.cos(7*np.pi*x_tr)
STEPS3 = 800 if FAST else 1500

W3, LS3 = 16, [1, 2, 4, 6]
trained_counts = []
print("매끄러운 목표 학습 (폭 16)")
print("  L   학습 전 조각   학습 후 조각   최종 MSE")
for L in LS3:
    Ws, bs = init_net(L, W3, np.random.default_rng(SEED+1))
    c0 = count_regions(Ws, bs, XS)
    mse, _ = train(Ws, bs, x_tr, y_smooth, STEPS3)
    c1 = count_regions(Ws, bs, XS)
    trained_counts.append(c1)
    print(f" {L:>2}   {c0:>11}   {c1:>11}   {mse:.5f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
LL = np.array(LS)
ax.semilogy(LL, [(W3+1)**L for L in LS], 'k--', lw=1.2,
            label=lab(f'이론 최대 $(w{{+}}1)^L$, $w$={W3}', f'upper bound $(w+1)^L$, $w$={W3}'))
ax.semilogy(range(1, LMAX_PLANT+1), plant_counts, '^-', color=CB[4], ms=5,
            label=lab('심은 톱니 $2^L$ (폭 2)', 'planted sawtooth $2^L$ (width 2)'))
ax.semilogy(LL, init_counts[W3], 'o-', color=CB[2], ms=5,
            label=lab(f'무작위 초기화 (폭 {W3})', f'random init (width {W3})'))
ax.semilogy(LS3, trained_counts, 's-', color=CB[3], ms=5,
            label=lab(f'학습 후 (폭 {W3})', f'after training (width {W3})'))
ax.semilogy(LL, W3*LL, ':', color=CB[6], lw=1.4,
            label=lab('참고: 총 유닛 수 $wL$', 'reference: total units $wL$'))
ax.set_xlabel(lab('깊이 $L$ (은닉층 수)', 'depth $L$'))
ax.set_ylabel(lab('선형 조각 수', 'number of linear regions'))
ax.set_title(lab('지수적 표현력은 설계했을 때만 나타난다',
                 'exponential expressivity appears only when designed for'), fontsize=10)
ax.legend(fontsize=8, loc='upper left')
show('region_counts')

> ### 이 그림이 §1.8.4의 재료다
>
> 검은 점선(이론 최대)과 파란 선(실제 망) 사이가 **수십 자릿수** 벌어진다.
> 학습해도 초록 선은 점선(총 유닛 수)을 크게 넘지 못한다.
>
> §1.8.1–1.8.2는 **깊이가 지수적 표현력을 준다**고 증명했고, 그 증명은 옳다.
> 그러나 **우리가 실제로 얻는 망은 그 표현력을 쓰지 않는다.**
> "무엇이 가능한가"와 "무엇이 일어나는가"가 다르다는 것 — 6부의 주제가 여기서 처음 보인다.

---
## 4. §1.7.6의 회수 — 표현할 수 있는데 학습하지 못한다

이제 결정적인 실험이다. 목표 함수를 $T_L$ 자체로 두고, 깊이 6·폭 32인 망에게 학습시킨다.

**이 망은 $T_L$ 을 표현할 수 있다** ($L \le 6$ 이면). 1.1절에서 그 가중치를 실제로 적었고,
필요한 것은 층당 유닛 2개뿐이다. 폭 32는 그보다 16배 넉넉하다.

곧 **$T_L \in H$ 이고 손실 0인 해가 존재한다.** 경사하강이 그것을 찾는지 본다.

In [ ]:
L_NET, W_NET = 6, 32
STEPS4 = 1200 if FAST else 2500
x4 = np.sort(np.random.default_rng(0).uniform(0, 1, 500 if FAST else 800))
LS4 = [1, 2, 3, 4, 5, 6]

print(f"망: 깊이 {L_NET} · 폭 {W_NET} (표현에 필요한 폭은 2)")
print("  L    var(T_L)    학습 MSE     설명력    심은 해의 MSE")
res4 = []
for L in LS4:
    y4 = T_ref(x4, L); v = float(np.var(y4))
    Ws, bs = init_net(L_NET, W_NET, np.random.default_rng(SEED+2))
    mse, hist = train(Ws, bs, x4, y4, STEPS4)
    Wp, bp = plant_sawtooth(L)
    mse_plant = float(np.mean((forward(x4, Wp, bp) - y4)**2))
    res4.append((L, v, mse, 1 - mse/v, mse_plant, hist, (Ws, bs)))
    print(f" {L:>2}    {v:.4f}     {mse:.5f}    {1-mse/v:6.1%}    {mse_plant:.1e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))

axes[0].plot([r[0] for r in res4], [100*r[3] for r in res4], 'o-', color=CB[4], ms=6)
axes[0].axhline(0, color=CB[0], lw=0.8, ls=':')
axes[0].set_xlabel(lab('목표의 깊이 $L$ (즉 $T_L$)', 'target depth $L$'))
axes[0].set_ylabel(lab('설명력 (%)', 'variance explained (%)'))
axes[0].set_ylim(-5, 105)
axes[0].set_title(lab('심은 해의 손실은 모든 $L$에서 0이다',
                      'the planted solution has zero loss for every $L$'), fontsize=10)

Lshow = 6
r = [q for q in res4 if q[0] == Lshow][0]
xz2 = np.linspace(0, 1, 3001)
axes[1].plot(xz2, T_ref(xz2, Lshow), color=CB[0], lw=1.0,
             label=lab(f'목표 $T_{{{Lshow}}}$', f'target $T_{{{Lshow}}}$'))
axes[1].plot(xz2, forward(xz2, *r[6]), color=CB[4], lw=1.4,
             label=lab('학습 결과', 'learned'))
axes[1].set_xlabel('$x$'); axes[1].set_ylabel('$y$')
axes[1].set_title(lab(f'$L$ = {Lshow}: 설명력 {100*r[3]:.0f}%',
                      f'$L$ = {Lshow}: {100*r[3]:.0f}% explained'), fontsize=10)
axes[1].legend(fontsize=8)
show('sawtooth_learning_failure')

> ### 이것이 §1.4.1의 구분이 아픈 자리다
>
> $T_5$ 와 $T_6$ 는 이 망의 가설 공간 $H$ 안에 있고, 손실을 정확히 0으로 만드는 가중치를
> 우리가 **손으로 적을 수 있다.** 그런데 경사하강은 거기 도달하지 못한다.
>
> $$T_L \in H \qquad \text{그러나} \qquad T_L \notin H_{\mathrm{reach}}$$
>
> §1.7.6에서 예고한 대로, **표현 가능성과 학습 가능성은 다르다.**
> 그리고 §1.7.6 3절의 직관이 여기 그대로 적용된다 — $T_L$ 은 국소적으로 정보가 없고
> 합성의 전체 구조에서만 신호가 나오므로, 국소적·평균적 정보인 기울기가 방향을 잡지 못한다.

---
## 5. 자기 점검

1. 2절에서 조각 수가 총 유닛 수 $wL$ 근처에 머물렀다. **폭을 늘리는 것과 깊이를 늘리는 것 중** 어느 쪽이 조각 수를 더 늘리는가? 표에서 확인하라.
2. 1.1절의 심은 톱니에서 **기울기의 절댓값**은 얼마인가? (§1.8.1 정리) $L = 20$이면 얼마가 되며, 그것이 학습에 어떤 문제를 일으키겠는가?
3. 4절에서 $L$ 이 커질수록 실패하는데, **망의 깊이는 6으로 고정**되어 있었다. 무엇이 어려워진 것인가 — 표현인가 최적화인가?
4. 4절의 목표를 $T_6$ 대신 $T_6$ 에 작은 잡음을 더한 것으로 바꾸면 결과가 나아지겠는가? 나빠지겠는가?

In [ ]:
# 자기 점검 2의 확인 — 심은 톱니의 기울기
for L in [1, 4, 8, 11]:
    Ws, bs = plant_sawtooth(L)
    y = forward(XS, Ws, bs)
    s = np.abs(np.diff(y)/np.diff(XS))
    print(f"  L={L:>2}: 기울기 절댓값 중앙값 {np.median(s):>8.1f}   이론 2^L = {2**L}")
print("\n-> 깊이로 얻은 표현력에는 기울기의 지수적 증가가 따라온다 (§1.8.5 · 2부).")

---
## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `NGRID` | 0절 | 100,001 | 조각 세기 해상도. 낮추면 좁은 조각을 놓쳐 **과소 계수** |
| `WS_LIST`, `LS` | 2절 | [4,16], 1~8 | 무작위 초기화 실험의 폭·깊이 |
| `bstd` | 1절 | 0.5 | 편향 초기화. **0으로 두면 1차원에서 모든 꺾임이 원점에 겹쳐 조각이 2개뿐** |
| `L_NET`, `W_NET` | 4절 | 6, 32 | 학습할 망의 크기. 폭을 128로 늘려도 $T_6$ 은 여전히 안 된다 |
| `STEPS4` | 4절 | 2500 | 학습 걸음 수 |
| `SAVE_PDF` | 0절 | False | 그림을 벡터 PDF로 저장 |

**권하는 첫 실험** — 1절의 `bstd` 를 `0.0` 으로 두고 2절을 다시 실행하십시오. 조각 수가 깊이·폭과
무관하게 **2개**로 고정됩니다. 1차원 입력에서 편향이 없으면 모든 층의 꺾임이 원점 하나에 겹치기 때문이며,
**초기화가 표현력에 직접 영향을 준다**는 것을 한 줄로 확인하는 방법입니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")